In [1]:
import matplotlib.pyplot as pl
import numpy as np
import pandas as pd
import xarray as xr
import math
import pickle

from fair import FAIR
from fair.interface import fill, initialise
from fair.io import read_properties
from pygments.lexers.textfmts import TodotxtLexer

import statsmodels.api as sm
import pmdarima as pm


In [2]:
with open("/glade/work/stevenxu/FAIR_models/origional_model_extension.pkl", "rb") as f_in:
    f1 = pickle.load(f_in)

Convergence year: 2023

In [5]:
no_emission_species = []
for specie in f1.species:
    da = f1.emissions.sel(scenario = f1.scenarios[0], specie=specie).mean(dim="config")
    if da.sum() == 0:
        no_emission_species.append(specie)

no_emission_species

['Solar',
 'Volcanic',
 'Aerosol-radiation interactions',
 'Aerosol-cloud interactions',
 'Ozone',
 'Light absorbing particles on snow and ice',
 'Stratospheric water vapour',
 'Land use',
 'Equivalent effective stratospheric chlorine']

In [ ]:
org_specie = f1.species
species_with_emissions = [s for s in org_specie if s not in no_emission_species]
arima_list = {}
count = 0

for specie in species_with_emissions:
    count += 1
    print("Fitting ARIMA for ", specie, " ", count, "/", len(species_with_emissions))
    da = f1.emissions.sel(scenario = f1.scenarios[5], specie=specie, timepoints = slice(None, 2023)).mean(dim="config")
    model = pm.auto_arima(da.values, start_p=1, start_q=1,
                                max_p=3, max_q=3, m=12,
                                start_P=0, seasonal=True,
                                d=1, D=1, trace=True,
                                error_action='ignore',  # don't want to know if an order does not work
                                suppress_warnings=True,  # don't want convergence warnings
                                stepwise=True)  # set to stepwise
    arima_list[specie] = model

arima_list


In [8]:
save_path = "/glade/work/stevenxu/FAIR_models/arima_list_emission_extension.pkl"
with open(save_path, "wb") as f:
    pickle.dump(arima_list, f)

### Improved model

In [ ]:
org_specie = f1.species
species_with_emissions = [s for s in org_specie if s not in no_emission_species]
arima_list = {}

for count, specie in enumerate(species_with_emissions, 1):
    print(f"Fitting ARIMA for {specie} ({count}/{len(species_with_emissions)})")
    
    # Extract the time series data
    da = f1.emissions.sel(scenario=f1.scenarios[5], specie=specie, timepoints=slice(None, 2023)).mean(dim="config")
    
    # Fit the non-seasonal ARIMA model
    model = pm.auto_arima(da.values, 
                          start_p=1, start_q=1,
                          max_p=5, max_q=5,       # Increased slightly to give the non-seasonal model more room
                          seasonal=False,         # <--- CHANGED: Turned off seasonality
                          d=None,                 # <--- CHANGED: Let auto_arima test for stationarity
                          trace=True,
                          error_action='ignore',  # don't want to know if an order does not work
                          suppress_warnings=True, # don't want convergence warnings
                          stepwise=True)          # set to stepwise
                          
    arima_list[specie] = model

arima_list

In [10]:
save_path = "/glade/work/stevenxu/FAIR_models/arima_list_emission_extension2.pkl"
with open(save_path, "wb") as f:
    pickle.dump(arima_list, f)